In [0]:
from pyspark.sql import functions as F

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df_silver = spark.read.format("delta").load(silver_path)

print("Silver customer count:", df_silver.count())

df_silver.orderBy("customer_id").show(truncate=False)

## Create gold_customers

In [0]:
from pyspark.sql import functions as F

gold_customers_path = (
    "abfss://gold@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/gold_customers/"
)

df_gold_customers = (
    df_silver
    .select(
        "customer_id",
        "customer_name",
        "city",
        "updated_at"
    )
    .dropDuplicates(["customer_id"])
)

(
    df_gold_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_customers_path)
)

print("Gold customer count:", df_gold_customers.count())

df_gold_customers.orderBy("customer_id").show(truncate=False)

## Create gold_city_summary

In [0]:
from pyspark.sql import functions as F

gold_city_summary_path = (
    "abfss://gold@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/gold_city_summary/"
)

df_gold_city_summary = (
    df_gold_customers
    .groupBy("city")
    .agg(
        F.countDistinct("customer_id").alias("customer_count")
    )
    .orderBy("city")
)

(
    df_gold_city_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_city_summary_path)
)

print("Gold city summary:")
spark.read.format("delta").load(gold_city_summary_path) \
    .orderBy("city") \
    .show(truncate=False)

In [0]:
# Gold layer Validation
from pyspark.sql import functions as F

gold_customers_path = (
    "abfss://gold@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/gold_customers/"
)

gold_city_summary_path = (
    "abfss://gold@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/gold_city_summary/"
)

customers = spark.read.format("delta").load(gold_customers_path)
cities = spark.read.format("delta").load(gold_city_summary_path)

print("Gold customer count:", customers.count())
print(
    "Duplicate customer IDs:",
    customers.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(
    "Deleted customer 4 in Gold:",
    customers.filter(F.col("customer_id") == 4).count()
)
print("Total customers across city summary:", cities.agg(
    F.sum("customer_count")
).first()[0])

assert customers.count() == 4
assert customers.groupBy("customer_id").count().filter(
    F.col("count") > 1
).count() == 0
assert customers.filter(F.col("customer_id") == 4).count() == 0
assert cities.agg(F.sum("customer_count")).first()[0] == 4

print("PASS: Gold validation successful.")